# 01 — Data Pipeline (Data Engineer owns)

Prepares every data artifact the agent needs **before** it can answer a question. Run top to bottom. Imports the building blocks from `src/nrcs_navigator/data`.

**Persistence:** one PostgreSQL database with the pgvector extension holds the `payment_rates` table and the eCFR embeddings (and later the agent checkpointer).

**Sources**
1. NRCS Practice FIPS CSV (FY2023–FY2025) → `payment_rates` table → powers `payment_estimator`
2. Four eCFR PDFs (Parts 1466, 1468, 1470, 1464) → embedded into the pgvector store for `eligibility_screener`

The two live scrape sources (practice standards, ranking dates) are fetched at query time, not here.

## Setup

Load config (incl. DATABASE_URL) and initialize the database: create the pgvector extension and the payment_rates schema.

In [ ]:
# from nrcs_navigator import config
# from nrcs_navigator.data import db
# db.init_db()  # CREATE EXTENSION IF NOT EXISTS vector; create payment_rates

## 1. Load the NRCS Practice FIPS CSV into payment_rates

In [1]:
from nrcs_navigator.data import fips_payments

# Read the raw FIPS CSV, normalize columns/types, and filter to in-scope
# programs. Rows are state-level (FY2023-2025). Writing these into the
# Postgres payment_rates table happens once db.py is implemented (next step).
payments = fips_payments.load_clean()
print(f"{len(payments):,} rows, {payments.shape[1]} columns")
payments.head()

19,285 rows, 8 columns


,state,program,practice_code,practice_name,fiscal_year,instance_count,dollars_obligated,avg_payment_per_instance
0,Alabama,CSP-GCI,E300GCI,Grassland Conservation Initiative,2023,95,52961,557.48
1,Alabama,CSP-GCI,E300GCI,Grassland Conservation Initiative,2024,70,66915,955.93
2,Alabama,CStwP Farm Bill,E449C,"Advanced Automated IWM - Year 2-5, soil moistu...",2025,5,12160,2432.00
3,Alabama,CStwP Farm Bill,314,Brush Management,2023,28,11584,413.71
4,Alabama,CStwP Farm Bill,314,Brush Management,2025,16,272,17.00


## 2. Download, extract, and chunk the eCFR regulation PDFs

In [ ]:
# Fetch the 4 eCFR PDFs into data/raw, extract text, chunk with metadata.
# from nrcs_navigator.data import ecfr_loader

## 3. Embed chunks into the pgvector store

In [ ]:
# Embed the eCFR chunks and upsert them into the PGVector collection in Postgres.
# from nrcs_navigator.data import vectorstore
# vectorstore.build_index(chunks)

## 4. Smoke check

Confirm payment_rates queries return rows and the pgvector store returns matches.

In [ ]:
# Quick sanity check: one SQL payment lookup and one similarity search.